# PBS Utah data preparation

Source exploration, quality checks and conservative cleanup for the [EDA notebook](analysis.ipynb). Original exports remain unchanged. Running the cleanup cells creates a new versioned output directory under `data/cleaned/`.

Run cells in order from the project root. Raw exports are required. The shared source copy is in `notebooks/data_preparation.ipynb`. The reproduction command executes cleanup cells by their stable IDs; retain those IDs.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'unite_payments.txt').exists():
    ROOT = ROOT.parent
FILES = sorted(p for p in ROOT.iterdir() if p.suffix in {'.csv', '.txt'})
tables = {p.stem: pd.read_csv(p, dtype=str, keep_default_na=False, encoding='utf-8-sig') for p in FILES}
print('Loaded', len(tables), 'comma-delimited files; identifiers retained as strings.')

## Inventory and duplicate rows

An exact duplicate count reports excess rows beyond the first occurrence. A unique key check is separate from a whole-row duplicate check.

In [ ]:
inventory = pd.DataFrame([{'file': p.name, 'rows': len(tables[p.stem]), 'columns': len(tables[p.stem].columns), 'duplicate_excess': int(tables[p.stem].duplicated().sum())} for p in FILES])
print(inventory.to_string(index=False))
print('Total source rows:', int(inventory.rows.sum()))

## Column quality

Blank means an empty or whitespace-only source value. Literal values such as `Unknown` remain separate from blanks. Constant fields and empty fields can be omitted from analytical views while retained in raw exports.

In [ ]:
quality = pd.DataFrame([{'table': name, 'column': col, 'blank': int(df[col].str.strip().eq('').sum()), 'blank_pct': round(100 * df[col].str.strip().eq('').mean(), 2), 'distinct_nonblank': df.loc[df[col].str.strip().ne(''), col].nunique(), 'outer_whitespace': int(df[col].ne(df[col].str.strip()).sum())} for name, df in tables.items() for col in df])
print('Entirely empty fields:')
print(quality.loc[quality.blank_pct.eq(100), ['table', 'column']].to_string(index=False))
print('\nConstant nonblank fields:', int(quality.distinct_nonblank.eq(1).sum()))
print('Values with outer whitespace:', int(quality.outer_whitespace.sum()))
print('\nConstituent missingness:')
print(quality.loc[quality.table.eq('constituents_w_memberships') & quality.blank.gt(0), ['column','blank','blank_pct']].to_string(index=False))

In [ ]:
c = tables['constituents_w_memberships']
print('Constituent Type equals Primary Constituent Type:', c['Constituent Type'].equals(c['Primary Constituent Type']))
print('Nonblank constituent city values:', c.loc[c['Address City'].ne(''), 'Address City'].unique().tolist())
print('Constituent ZIP lengths:', c['Address ZIP'].str.len().value_counts().sort_index().to_dict())
print('Constituent types:', c['Constituent Type'].value_counts().to_dict())
print('Nonblank degree counts:', c.loc[c['UofU Degree Count'].ne(''), 'UofU Degree Count'].unique().tolist())

Synthetic city values and constant source/campaign codes are placeholders, not meaningful geography or segmentation. Uniform zero degree counts alongside alumni categories also require clarification. Missing marital status does not imply unmarried; missing donor class does not imply non-donor.

## Keys and relationship coverage

In [ ]:
keys = {'constituents_w_memberships':['Constituent ID'], 'campaign_codes':['Marketing Code'], 'officer_portfolios':['portfolio_id'], 'cultivation':['synthetic_id','fiscal_year'], 'passport_viewing':['constituent_id','fiscal_year','genre'], 'unite_payments':['payment_id'], 'soft_credits':['Credit ID']}
key_checks = pd.DataFrame([{'table': name, 'key': ' + '.join(key), 'duplicate_excess': int(tables[name].duplicated(key).sum()), 'blank_key_rows': int(tables[name][key].eq('').any(axis=1).sum())} for name,key in keys.items()])
print(key_checks.to_string(index=False))
master = set(c['Constituent ID']) - {''}
relations = []
for name, df in tables.items():
    for col in df:
        if col in ['donor_id','constituent_id','synthetic_id','Spouse Constituent ID','Soft Credit Constituent ID'] or col.startswith('constituent_id_'):
            populated = df[col].ne('')
            unmatched = populated & ~df[col].isin(master)
            relations.append({'table':name,'column':col,'populated_rows':int(populated.sum()),'unmatched_rows':int(unmatched.sum()),'unmatched_ids':df.loc[unmatched,col].nunique()})
print('\nConstituent relationship checks:')
print(pd.DataFrame(relations).to_string(index=False))

## Payments, soft credits and unsafe joins

Soft credits represent recognition or attribution, not additional cash revenue. Preserve orphan payments and their amounts with an unresolved relationship flag. A designation is not a transaction key.

In [ ]:
u = tables['unite_payments']
s_raw = tables['soft_credits']
s = s_raw.drop_duplicates()
legacy = tables['team_approach_legacy_payments']
for name, df, amount, date in [('Unite',u,'payment_amount','credit_date'),('Legacy',legacy,'payment_amount','gift_date'),('Soft credits raw',s_raw,'Credit Amount','Credit Date')]:
    amounts = pd.to_numeric(df[amount], errors='coerce')
    dates = pd.to_datetime(df[date], format='mixed', errors='coerce')
    print(name, {'rows':len(df),'amount_total':float(amounts.sum()),'invalid_amounts':int(amounts.isna().sum()),'zero_amounts':int(amounts.eq(0).sum()),'negative_amounts':int(amounts.lt(0).sum()),'invalid_dates':int(dates.isna().sum()),'first_date':str(dates.min().date()),'last_date':str(dates.max().date())})
print('Soft duplicate excess:',len(s_raw)-len(s))
print('Soft duplicate amount:',pd.to_numeric(s_raw.loc[s_raw.duplicated(),'Credit Amount']).sum())
print('Deduplicated soft total:',pd.to_numeric(s['Credit Amount']).sum())
print('Unmatched Unite donors:',u.loc[~u.donor_id.isin(master),'donor_id'].value_counts().to_dict())
print('Unmatched deduplicated soft recipients:',s.loc[~s['Soft Credit Constituent ID'].isin(master),'Soft Credit Constituent ID'].value_counts().to_dict())

In [ ]:
designation_counts = u.designation_detail_id.value_counts()
print('Rows a raw designation-only inner join would produce:',int(s_raw['Designation Detail ID'].map(designation_counts).fillna(0).sum()))
print('Maximum payments per designation:',int(designation_counts.max()))
match_keys = ['donor_id','designation_detail_id','credit_date','payment_amount']
candidate_counts = u.groupby(match_keys).size()
credit_keys = s[['Hard Credit Org ID','Designation Detail ID','Credit Date','Credit Amount']].set_axis(match_keys,axis=1)
match_counts = candidate_counts.reindex(pd.MultiIndex.from_frame(credit_keys)).fillna(0)
print('Candidate matches using organization + designation + date + amount:', {'none':int(match_counts.eq(0).sum()),'one':int(match_counts.eq(1).sum()),'multiple':int(match_counts.gt(1).sum()),'maximum':int(match_counts.max())})

Request a source `payment_id` on soft-credit exports. The four-field match above is only a diagnostic: 61 credits remain ambiguous and 34 have no candidate. Do not allocate these automatically. Date ranges are separate across the two payment systems, but this does not prove completeness or exclude migration artifacts.

In [ ]:
m = tables['campaign_codes']
known_codes = set(m['Marketing Code'])
unmapped = ~u.marketing_code.isin(known_codes)
print('Payments with missing campaign mappings:',int(unmapped.sum()))
print('Distinct unmapped marketing codes:',u.loc[unmapped,'marketing_code'].nunique())
mapped_names = u.marketing_code.map(m.set_index('Marketing Code')['Campaign Name'])
print('Campaign name mismatches among mapped payments:',int((~unmapped & u.campaign_name.ne(mapped_names)).sum()))
link_columns = [col for col in legacy if col.startswith('constituent_id_')]
print('Legacy constituent links per payment row:',legacy[link_columns].ne('').sum(axis=1).value_counts().sort_index().to_dict())
print('Legacy account ID duplicate excess:',int(legacy.duplicated(['ta_account_id']).sum()))

Use a bridge table for legacy payment-to-constituent links. The 9,440 rows linked to two people must retain their cash amount once. An approved attribution rule is needed for donor-level totals. The 216,128 rows with no links should remain in source cash totals. `ta_account_id` is not a payment key.

## Viewing and cultivation grain

In [ ]:
v = tables['passport_viewing']
watch = pd.to_numeric(v.mean_percent_watched,errors='coerce')
titles = pd.to_numeric(v.titles_viewed,errors='coerce')
print('Viewing years:',v.fiscal_year.value_counts().sort_index().to_dict())
print('Watch fraction range:',float(watch.min()),float(watch.max()))
print('Invalid/out-of-range watch fractions:',int((watch.isna() | ~watch.between(0,1)).sum()))
print('Invalid/negative/noninteger title counts:',int((titles.isna() | titles.lt(0) | titles.mod(1).ne(0)).sum()))
a = tables['cultivation'].merge(tables['officer_portfolios'],on='portfolio_id',how='left',validate='many_to_one',indicator=True)
print('Unmatched cultivation portfolios:',int(a['_merge'].ne('both').sum()))
print('Cultivation years:',a.fiscal_year.value_counts().sort_index().to_dict())
print('Selected values:',a.selected.value_counts().to_dict())
print('contacted equals holdout:',int(a.contacted.eq(a.holdout).sum()))
print('Portfolio-year group size distribution:',a.groupby(['portfolio_id','fiscal_year']).size().value_counts().to_dict())
rollout = pd.to_numeric(a.rollout_fy,errors='coerce')
expected_adoption = pd.to_numeric(a.fiscal_year).ge(rollout)
print('Adoption mismatches against year >= rollout:',int(a.program_adopted.eq('True').ne(expected_adoption).sum()))
print('Portfolios with blank rollout year:',int(tables['officer_portfolios'].rollout_fy.eq('').sum()))

Aggregate viewing to constituent-year before joining annual giving or cultivation outcomes. A weighted watch mean requires confirmation that `titles_viewed` is the denominator behind each source mean; summed genre counts may not represent distinct titles.

Cultivation includes only selected records, with 125 rows per portfolio-year. Contact and holdout flags are exact opposites, and adoption follows rollout year exactly. Missing rollout is treated as not adopted only in the diagnostic above; its business meaning is unresolved. These patterns do not establish causal impact. Avoid assigning control status to absent rows or using current constituent attributes as historical predictors without dated snapshots.

## Execute conservative cleanup

The following cells write a new versioned directory under `data/cleaned/`. Raw exports and previous runs remain unchanged. Staging retains source fields with explicit types and provenance. Curated tables omit entirely empty fields and the confirmed duplicate constituent-type column. Soft credits are deduplicated; unresolved links are retained and flagged. No fiscal calendar, historical snapshot, soft-credit allocation or household rule is invented.

Requires pandas and pyarrow. Run all cells in order. Each run creates a new output directory.

In [ ]:
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
import hashlib
import json
import re
import uuid

# Every run gets a separate directory, so previous outputs remain recoverable.
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
OUT = ROOT / 'data' / 'cleaned' / RUN_ID
for folder in ['staging', 'curated', 'exceptions', 'metadata']:
    (OUT / folder).mkdir(parents=True, exist_ok=False)

def checksum(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

source_manifest = [
    {'file': p.name, 'sha256': checksum(p), 'bytes': p.stat().st_size,
     'rows': len(tables[p.stem]), 'delimiter': ',', 'encoding': 'utf-8-sig'}
    for p in FILES
]
source_hashes = {Path(x['file']).stem: x['sha256'] for x in source_manifest}
written = []
checks = []

def write_table(df, folder, name):
    path = OUT / folder / (name + '.parquet')
    df.to_parquet(path, index=False, engine='pyarrow', compression='snappy')
    written.append({'path': str(path.relative_to(OUT)), 'rows': len(df), 'columns': len(df.columns)})
    return path

def check(name, actual, expected):
    passed = bool(actual == expected)
    checks.append({'check': name, 'actual': str(actual), 'expected': str(expected), 'passed': passed})
    if not passed:
        raise AssertionError(f'{name}: {actual!r} != {expected!r}')

def snake(name):
    return re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')

# Decimal objects are exported as Arrow decimal columns. No float currency conversion.
def exact_amount(series):
    def convert(value):
        if pd.isna(value):
            return None
        number = Decimal(str(value))
        if not number.is_finite():
            raise ValueError('Non-finite monetary amount')
        return number
    return series.map(convert)

def amount_sum(series):
    return sum((x for x in series if x is not None and not pd.isna(x)), Decimal('0'))

print('Output directory:', OUT)


### Typed staging and data dictionary

In [ ]:
boolean_columns = {
    'constituents_w_memberships': ['has_uofu_legacy_gift', 'legacy_circle_member_pbs_utah', 'sustainer_pbs_utah'],
    'campaign_codes': ['active', 'solicitation'],
    'cultivation': ['selected', 'contacted', 'holdout', 'program_adopted'],
}
integer_columns = {
    'constituents_w_memberships': ['age', 'uofu_degree_count', 'latest_uofu_degree_year'],
    'officer_portfolios': ['capacity', 'rollout_fy'],
    'cultivation': ['fiscal_year'],
    'passport_viewing': ['fiscal_year', 'titles_viewed'],
}
money_columns = {
    'constituents_w_memberships': ['lifetime_uofu_fundraising', 'previous_fy_uofu_cash', 'prospect_committed_giving_amount_uofu'],
    'team_approach_legacy_payments': ['payment_amount'],
    'unite_payments': ['payment_amount'],
    'soft_credits': ['credit_amount'],
}
date_columns = {
    'team_approach_legacy_payments': ['gift_date'],
    'unite_payments': ['credit_date'],
    'soft_credits': ['credit_date'],
}
staged = {}
dictionary_rows = []
for name, raw in tables.items():
    names = [snake(col) for col in raw.columns]
    check(name + ': normalized column names are unique', len(set(names)), len(names))
    df = raw.set_axis(names, axis=1).copy()
    for col in df:
        df[col] = df[col].astype('string').str.strip().replace('', pd.NA)
    for col in boolean_columns.get(name, []):
        invalid = df[col].notna() & ~df[col].isin(['True', 'False'])
        check(name + '.' + col + ': valid booleans', int(invalid.sum()), 0)
        df[col] = df[col].map({'True': True, 'False': False}).astype('boolean')
    for col in integer_columns.get(name, []):
        values = pd.to_numeric(df[col], errors='raise')
        check(name + '.' + col + ': integral values', int((values.dropna() % 1 != 0).sum()), 0)
        df[col] = values.astype('Int64')
    for col in money_columns.get(name, []):
        df[col] = exact_amount(df[col])
    for col in date_columns.get(name, []):
        df[col] = pd.to_datetime(df[col], format='mixed', errors='raise').dt.normalize()
    if name == 'passport_viewing':
        df['mean_percent_watched'] = pd.to_numeric(df['mean_percent_watched'], errors='raise').astype('Float64')
        df = df.rename(columns={'mean_percent_watched': 'mean_watch_fraction'})
    df['source_row_number'] = pd.Series(range(1, len(df) + 1), dtype='Int64')
    df['source_file'] = next(p.name for p in FILES if p.stem == name)
    staged[name] = df
    write_table(df, 'staging', name)
    check(name + ': staging row count', len(df), len(raw))
    for original in raw.columns:
        normalized = snake(original)
        if name == 'passport_viewing' and normalized == 'mean_percent_watched':
            normalized = 'mean_watch_fraction'
        dictionary_rows.append({
            'table': name, 'source_column': original, 'clean_column': normalized,
            'pandas_dtype': str(df[normalized].dtype),
            'null_count': int(df[normalized].isna().sum()),
            'distinct_nonnull': int(df[normalized].nunique()),
            'all_null': bool(df[normalized].isna().all()),
        })
print('Staged', len(staged), 'tables with explicit types and source-row provenance.')
print('Blank strings converted to null; identifiers and ZIP codes retained as strings.')


### Entities and soft-credit deduplication

In [ ]:
constituents = staged['constituents_w_memberships'].copy()
check('Constituent Type and Primary Constituent Type are redundant',
      bool(constituents.constituent_type.equals(constituents.primary_constituent_type)), True)
constituents = constituents.drop(columns=['primary_constituent_type'])
master_ids = set(constituents.constituent_id.dropna())
curated = {}
removed_fields = []
def compact(name, df):
    # Empty source fields are retained in staging; omit only in curated views.
    empty = [col for col in df if df[col].isna().all()]
    removed_fields.extend({'table': name, 'column': col, 'reason': 'entirely null'} for col in empty)
    return df.drop(columns=empty)
removed_fields.append({'table':'constituents','column':'primary_constituent_type','reason':'identical to constituent_type'})
curated['constituents'] = compact('constituents', constituents)
campaigns = staged['campaign_codes'].copy()
curated['campaigns'] = compact('campaigns', campaigns)
curated['portfolios'] = staged['officer_portfolios'].copy()
assignments = staged['cultivation'].rename(columns={'synthetic_id':'constituent_id'}).copy()
assignments['constituent_resolved'] = assignments.constituent_id.isin(master_ids)
curated['cultivation_assignments'] = assignments
curated['viewing_annual_genre'] = staged['passport_viewing'].copy()

relationships = constituents.loc[constituents.spouse_constituent_id.notna(),
    ['constituent_id','spouse_constituent_id','source_row_number','source_file']].rename(
    columns={'spouse_constituent_id':'related_constituent_id'})
relationships['relationship_type'] = 'spouse'
relationships['related_constituent_resolved'] = relationships.related_constituent_id.isin(master_ids)
curated['constituent_relationships'] = relationships

soft = staged['soft_credits']
duplicate_mask = tables['soft_credits'].duplicated().to_numpy()
write_table(soft.loc[duplicate_mask].copy(), 'exceptions', 'soft_credit_duplicate_excess')
soft_clean = soft.loc[~duplicate_mask].copy()
soft_clean['constituent_resolved'] = soft_clean.soft_credit_constituent_id.isin(master_ids)
# A match is a review candidate, not a resolved payment linkage.
soft_clean['resolved_payment_id'] = pd.Series(pd.NA, index=soft_clean.index, dtype='string')
write_table(soft_clean.loc[~soft_clean.constituent_resolved], 'exceptions', 'soft_credit_unresolved_constituents')
write_table(curated['portfolios'].loc[curated['portfolios'].rollout_fy.isna()], 'exceptions', 'portfolio_missing_rollout')
check('Soft credits: removed rows reconcile', len(soft) - len(soft_clean), int(duplicate_mask.sum()))
check('Soft credits: amount reconciliation',
      amount_sum(soft.credit_amount) - amount_sum(soft_clean.credit_amount),
      amount_sum(soft.loc[duplicate_mask, 'credit_amount']))
check('Soft credits: remaining credit_id unique', int(soft_clean.credit_id.duplicated().sum()), 0)
print('Soft-credit excess duplicates removed:', int(duplicate_mask.sum()))
print('Soft-credit amount excluded:', amount_sum(soft.loc[duplicate_mask, 'credit_amount']))
print('Deduplicated soft-credit total:', amount_sum(soft_clean.credit_amount))


### Payment facts, link tables and review exceptions

In [ ]:
unite = staged['unite_payments'].copy()
unite['constituent_resolved'] = unite.donor_id.isin(master_ids)
unite['campaign_resolved'] = unite.marketing_code.isin(set(campaigns.marketing_code))
write_table(unite.loc[~unite.constituent_resolved], 'exceptions', 'unite_unresolved_constituents')
write_table(unite.loc[~unite.campaign_resolved], 'exceptions', 'unite_unmapped_campaigns')
write_table(unite.loc[unite.payment_amount.eq(Decimal('0'))], 'exceptions', 'unite_zero_amount_review')
legacy = staged['team_approach_legacy_payments'].copy()
# The checksum identifies an immutable batch, not the extraction date.
legacy['legacy_payment_id'] = (
    'ta:' + source_hashes['team_approach_legacy_payments'] + ':' +
    legacy.source_row_number.astype('string')
)
link_cols = [col for col in legacy if col.startswith('constituent_id_')]
link_count = legacy[link_cols].notna().sum(axis=1)
write_table(legacy.loc[link_count.eq(0)], 'exceptions', 'legacy_payments_without_constituent')
write_table(legacy.loc[link_count.gt(1)], 'exceptions', 'legacy_payments_multiple_constituents')
legacy_bridge = legacy[['legacy_payment_id','source_row_number','source_file'] + link_cols].melt(
    id_vars=['legacy_payment_id','source_row_number','source_file'],
    value_vars=link_cols, var_name='source_slot', value_name='constituent_id'
).dropna(subset=['constituent_id'])
legacy_bridge['constituent_resolved'] = legacy_bridge.constituent_id.isin(master_ids)
check('Legacy bridge: row + constituent pairs unique',
      int(legacy_bridge.duplicated(['legacy_payment_id','constituent_id']).sum()), 0)
curated['legacy_payment_constituents'] = legacy_bridge
curated['unite_payments'] = unite
curated['legacy_payments'] = compact('legacy_payments', legacy.drop(columns=link_cols))

# Conservative common schema: original classifications remain in source-specific tables.
u_pay = unite[['payment_id','credit_date','payment_amount','marketing_code','source_row_number','source_file']].rename(
    columns={'payment_id':'source_payment_id','credit_date':'source_date'})
u_pay = u_pay.copy()
u_pay['payment_key'] = 'unite:' + u_pay.source_payment_id
u_pay['source_system'] = 'unite'
u_pay['source_date_field'] = 'credit_date'
u_pay['campaign_resolved'] = unite.campaign_resolved.to_numpy()
l_pay = legacy[['legacy_payment_id','gift_date','payment_amount','source_row_number','source_file']].rename(
    columns={'legacy_payment_id':'source_payment_id','gift_date':'source_date'}).copy()
l_pay['payment_key'] = l_pay.source_payment_id
l_pay['source_system'] = 'team_approach'
l_pay['source_date_field'] = 'gift_date'
l_pay['marketing_code'] = pd.Series(pd.NA, index=l_pay.index, dtype='string')
l_pay['campaign_resolved'] = pd.Series(pd.NA, index=l_pay.index, dtype='boolean')
payments_clean = pd.concat([u_pay,l_pay], ignore_index=True)
curated['payments'] = payments_clean

u_links = unite[['donor_id','source_row_number','source_file','constituent_resolved']].rename(columns={'donor_id':'constituent_id'}).copy()
u_links['payment_key'] = 'unite:' + unite.payment_id
u_links['source_slot'] = 'donor_id'
l_links = legacy_bridge.rename(columns={'legacy_payment_id':'payment_key'})
curated['payment_constituents'] = pd.concat([u_links,l_links], ignore_index=True)
curated['payment_constituents']['attribution_weight'] = pd.Series(
    pd.NA, index=curated['payment_constituents'].index, dtype='Float64')

# Candidate matching never expands the soft-credit table.
group_keys = ['donor_id','designation_detail_id','credit_date','payment_amount']
candidate_lookup = unite.groupby(group_keys, dropna=False).agg(
    candidate_count=('payment_id','size'), candidate_payment_id=('payment_id','first')
).reset_index().rename(columns={
    'donor_id':'hard_credit_org_id', 'designation_detail_id':'designation_detail_id',
    'payment_amount':'credit_amount'})
candidate_lookup.loc[candidate_lookup.candidate_count.ne(1), 'candidate_payment_id'] = pd.NA
soft_clean = soft_clean.merge(candidate_lookup,
    on=['hard_credit_org_id','designation_detail_id','credit_date','credit_amount'],
    how='left',validate='many_to_one')
soft_clean['candidate_count'] = soft_clean.candidate_count.fillna(0).astype('Int64')
soft_clean['match_review_status'] = soft_clean.candidate_count.map(
    lambda n: 'no_candidate' if n == 0 else ('single_candidate_unconfirmed' if n == 1 else 'ambiguous'))
curated['soft_credits'] = soft_clean
write_table(soft_clean.loc[soft_clean.candidate_count.ne(1)], 'exceptions', 'soft_credit_unmatched_or_ambiguous')
write_table(soft_clean.loc[soft_clean.credit_amount.eq(Decimal('0'))], 'exceptions', 'soft_credit_zero_amount_review')
print('Payment rows retained:',len(payments_clean))
print('Unresolved Unite donor rows:',int((~unite.constituent_resolved).sum()))
print('Unmapped campaign rows:',int((~unite.campaign_resolved).sum()))
print('Legacy unlinked / multiple-linked rows:',int(link_count.eq(0).sum()),int(link_count.gt(1).sum()))
print('Soft-credit candidate statuses:',soft_clean.match_review_status.value_counts().to_dict())


### Reconciliation, persisted-file checks and manifest

In [ ]:
# Validate before marking this run complete.
check('Unified payment count',len(payments_clean),len(unite)+len(legacy))
check('Unified payment keys unique',int(payments_clean.payment_key.duplicated().sum()),0)
for source, df in [('unite',unite),('team_approach',legacy)]:
    result = payments_clean.loc[payments_clean.source_system.eq(source)]
    check(source + ': retained payment rows',len(result),len(df))
    check(source + ': exact amount reconciliation',amount_sum(result.payment_amount),amount_sum(df.payment_amount))
    check(source + ': parsed dates are complete',int(result.source_date.isna().sum()),0)
for table, key in [
    ('constituents',['constituent_id']),('campaigns',['marketing_code']),
    ('portfolios',['portfolio_id']),('cultivation_assignments',['constituent_id','fiscal_year']),
    ('viewing_annual_genre',['constituent_id','fiscal_year','genre']),
    ('soft_credits',['credit_id']),
    ('payment_constituents',['payment_key','constituent_id']),
]:
    frame = curated[table]
    check(table + ': key uniqueness',int(frame.duplicated(key).sum()),0)
    check(table + ': nonnull key',int(frame[key].isna().any(axis=1).sum()),0)
view = curated['viewing_annual_genre']
check('Viewing: fraction range',bool(view.mean_watch_fraction.between(0,1).all()),True)
check('Viewing: nonnegative counts',bool(view.titles_viewed.ge(0).all()),True)
check('Cultivation: portfolio coverage',bool(assignments.portfolio_id.isin(curated['portfolios'].portfolio_id).all()),True)
check('Cultivation: constituent coverage',bool(assignments.constituent_resolved.all()),True)
check('Soft-credit candidate matching preserves rows',len(soft_clean),len(soft)-int(duplicate_mask.sum()))
check('Soft-credit candidate matching preserves amounts',amount_sum(soft_clean.credit_amount),
      amount_sum(soft.credit_amount)-amount_sum(soft.loc[duplicate_mask,'credit_amount']))
check('Payment links reference valid payment keys',
      bool(curated['payment_constituents'].payment_key.isin(set(payments_clean.payment_key)).all()),True)
for name, frame in curated.items():
    write_table(frame, 'curated', name)

dictionary = pd.DataFrame(dictionary_rows)
dictionary.to_csv(OUT / 'metadata' / 'source_data_dictionary.csv',index=False)
pd.DataFrame(removed_fields).to_csv(OUT / 'metadata' / 'omitted_curated_fields.csv',index=False)

# Inspect persisted schemas and row counts, not only in-memory frames.
import pyarrow.parquet as pq
for item in written:
    parquet = pq.ParquetFile(OUT / item['path'])
    check(item['path'] + ': saved row count',parquet.metadata.num_rows,item['rows'])
for item in source_manifest:
    check(item['file'] + ': original checksum unchanged',checksum(ROOT / item['file']),item['sha256'])
validation = pd.DataFrame(checks)
validation.to_csv(OUT / 'metadata' / 'validation_checks.csv',index=False)
notes = {
    'currency': 'Unconfirmed; monetary columns preserve source values as exact decimals.',
    'fiscal_calendar': 'Unconfirmed. No fiscal years are inferred from payment dates.',
    'snapshot_date': 'Not provided. Current constituent attributes are not historical features.',
    'soft_credit_linkage': 'Candidates only; resolved_payment_id remains null pending source IDs or review.',
    'allocation': 'Payment-to-constituent weights are null pending a business rule.',
    'raw_storage': 'Original exports stay in the project root; SHA-256 hashes recorded.',
    'legacy_keys': 'sha256(source file) plus source row number; stable for an unchanged export.',
    'synthetic_fields': 'Synthetic placeholders retained as source metadata; not imputed.',
}
manifest = {
    'run_id': RUN_ID, 'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'sources': source_manifest, 'outputs': written, 'decisions': notes,
    'validation_passed': bool(validation.passed.all()),
    'validation_check_count': len(validation),
    'reconciliation': {
        'unite_amount': str(amount_sum(unite.payment_amount)),
        'legacy_amount': str(amount_sum(legacy.payment_amount)),
        'soft_credit_raw_amount': str(amount_sum(soft.credit_amount)),
        'soft_credit_deduplicated_amount': str(amount_sum(soft_clean.credit_amount)),
        'soft_credit_duplicate_excess': int(duplicate_mask.sum()),
    },
}
(OUT / 'metadata' / 'manifest.json').write_text(json.dumps(manifest,indent=2) + '\n')
(OUT / '_SUCCESS').write_text('All validation checks passed. See metadata/manifest.json.\n')
print('Saved curated tables:',len(curated))
print('Saved exception tables:',sum(x['path'].startswith('exceptions/') for x in written))
print('Validation checks passed:',len(validation))
print('Output:',OUT)
print(pd.DataFrame(written).to_string(index=False))


### Verify exported monetary values

Read back only monetary columns to confirm that Parquet preserves exact decimal types and reconciled totals.

In [ ]:
# Verify decimal totals from the persisted curated files.
import pyarrow.compute as pc
from decimal import Decimal
saved_manifest = json.loads((OUT / 'metadata' / 'manifest.json').read_text())
saved_payments = pq.read_table(OUT / 'curated' / 'payments.parquet', columns=['payment_amount'])
saved_soft = pq.read_table(OUT / 'curated' / 'soft_credits.parquet', columns=['credit_amount'])
assert pd.api.types.is_object_dtype(pd.Series([saved_payments.column('payment_amount')[0].as_py()]).dtype)
assert isinstance(saved_payments.column('payment_amount')[0].as_py(), Decimal)
payment_total = pc.sum(saved_payments.column('payment_amount')).as_py()
soft_total = pc.sum(saved_soft.column('credit_amount')).as_py()
expected_payment_total = Decimal(saved_manifest['reconciliation']['unite_amount']) + Decimal(saved_manifest['reconciliation']['legacy_amount'])
assert payment_total == expected_payment_total
assert soft_total == Decimal(saved_manifest['reconciliation']['soft_credit_deduplicated_amount'])
assert (OUT / '_SUCCESS').exists()
print('Persisted cash total:', payment_total)
print('Persisted deduplicated soft-credit total:', soft_total)
print('Decimal type and exact totals verified after Parquet round trip.')
print('Ready for next step:', OUT / 'curated')
